# 02. mask와 latent prediction 실습

목표: 2차원 patch grid에서 context/target을 만들고, 예측 embedding과 target embedding 사이의 Smooth L1 loss 및 EMA를 계산한다. 실제 I-JEPA가 아닌 개념용 toy implementation이다.

In [ ]:
import random

def rectangle(top, left, height, width):
    # 직사각형 안의 patch 좌표를 set으로 만들면 겹침 검사가 쉽다.
    return {
        (row, col)
        for row in range(top, top + height)
        for col in range(left, left + width)
    }

grid_size = 8
target = rectangle(top=1, left=5, height=2, width=2)
context = rectangle(top=3, left=0, height=4, width=4)
assert context.isdisjoint(target), "context와 target이 겹칩니다"

for row in range(grid_size):
    symbols = []
    for col in range(grid_size):
        cell = (row, col)
        symbols.append("C" if cell in context else "T" if cell in target else ".")
    print(" ".join(symbols))

In [ ]:
def smooth_l1(prediction, target, beta=1.0):
    """작은 오차는 제곱, 큰 오차는 절댓값으로 다루는 평균 loss다."""
    losses = []
    for predicted, expected in zip(prediction, target):
        error = abs(predicted - expected)
        if error < beta:
            losses.append(0.5 * error * error / beta)
        else:
            losses.append(error - 0.5 * beta)
    return sum(losses) / len(losses)

target_embedding = [0.8, -0.2, 0.5, 1.1]
early_prediction = [0.1, 0.3, -0.4, 0.2]
better_prediction = [0.7, -0.1, 0.4, 0.9]
print("초기 loss:", smooth_l1(early_prediction, target_embedding))
print("개선 loss:", smooth_l1(better_prediction, target_embedding))

In [ ]:
def ema_update(target_weights, online_weights, momentum):
    # target branch에는 역전파하지 않고 online encoder를 천천히 따라가게 한다.
    return [
        momentum * target + (1.0 - momentum) * online
        for target, online in zip(target_weights, online_weights)
    ]

target_weights = [0.0, 0.0, 0.0]
online_history = [[1.0, 0.5, -0.5], [0.8, 1.0, 0.0], [1.2, 0.7, 0.3]]
for step, online_weights in enumerate(online_history, start=1):
    target_weights = ema_update(target_weights, online_weights, momentum=0.9)
    print(f"step {step}: {target_weights}")

## 응용 과제

- target 크기를 1×1과 4×4로 바꾸고 무엇이 너무 쉽거나 어려운지 설명한다.
- EMA momentum을 0.5, 0.9, 0.99로 바꾸고 target의 지연을 비교한다.
- 이미지가 아닌 표라면 직사각형 대신 어떤 feature subset mask가 적절할지 설계한다.